# NB-02 | Component Extraction & Data Flow Mapping
**Pipeline Step 2 of 5**

Uses GPT-4o to extract structured security components, data flows, and trust boundaries from the repository surface.

**Key improvements over v1:**
- Grounding check uses exact path matching against `valid_paths` (full relative paths), not bidirectional substring matching — eliminates false-positive grounded verdicts
- Ungrounded components are dropped *before* being saved; they do not propagate to NB03
- JSON schema validation ensures all required keys are present before saving
- Retry logic appends a corrective message on JSON parse failures instead of replaying the same bad prompt

**Input:** `repo_surface.json`  
**Output:** `components.json`

In [ ]:
!pip install -q openai tiktoken

import os, sys
sys.path.insert(0, ".")

from pipeline_utils import (
    load_json, save_json, get_logger,
    parse_json_response, call_with_retry,
)

import json
from pathlib import Path
from openai import OpenAI
import tiktoken

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
log    = get_logger("NB02")

## 2.1 — Load Surface

In [ ]:
surface = load_json(
    "repo_surface.json",
    schema_keys=["repo", "files", "valid_paths", "run_id"],
)

valid_paths: set[str] = set(surface["valid_paths"])   # full relative paths
RUN_ID = surface["run_id"]

log.info("Repo        : %s", surface["repo"])
log.info("Files       : %d", surface["file_count"])
log.info("Valid paths : %d", len(valid_paths))
log.info("OpenAPI     : %s", surface["openapi"] is not None)

## 2.2 — Build Code Context

Packs priority files into the token budget in the order they appear in `repo_surface.json` (already sorted by priority score by NB01).

In [ ]:
TOKEN_BUDGET = 12_000

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))


def build_code_context(files: dict[str, str], token_budget: int) -> str:
    """
    Pack as many files as fit within token_budget into a single code context string.

    Files are iterated in the order provided, which NB01 guarantees is
    descending priority score order.
    """
    chunks: list[str] = []
    used = 0
    for path, content in files.items():
        snippet = f"### FILE: {path}\n{content}\n"
        toks    = count_tokens(snippet)
        if used + toks > token_budget:
            log.info("Token budget reached at %d files (%d tokens used)", len(chunks), used)
            break
        chunks.append(snippet)
        used += toks
    log.info("Packed %d files into context (%d tokens)", len(chunks), used)
    return "\n".join(chunks)


code_ctx = build_code_context(surface["files"], TOKEN_BUDGET)

## 2.3 — Component Extraction via LLM

In [ ]:
SYSTEM = """
You are an expert AppSec architect. Analyze the codebase and extract a structured
security architecture. Respond ONLY with valid JSON — no markdown fences, no explanation.

Schema:
{
  "components": [
    {
      "id": "snake_case_unique_id",
      "name": "Human-readable name",
      "type": "service|database|auth|external_api|queue|cache|frontend",
      "description": "One sentence describing what this component does",
      "files": ["exact/relative/path/as/it/appears/in/the/codebase"],
      "data_handled": ["list of data types this component processes"]
    }
  ],
  "data_flows": [
    {
      "from": "component_id",
      "to": "component_id",
      "data": "what data is transferred",
      "protocol": "HTTP|HTTPS|SQL|gRPC|AMQP|Redis|etc",
      "authenticated": true
    }
  ],
  "trust_boundaries": [
    {
      "id": "unique_id",
      "name": "boundary name",
      "description": "what this boundary separates",
      "components_inside": ["component_ids"],
      "entry_points": ["list of entry point descriptions"]
    }
  ],
  "external_actors": [
    {
      "id": "unique_id",
      "name": "actor name",
      "trust_level": "untrusted|semi-trusted|trusted"
    }
  ]
}

CRITICAL: In 'files', use the EXACT relative paths visible in the code context
(e.g. "fastapi/routing.py", not just "routing.py" or an invented path).
"""


def _build_user_message(code_ctx: str, openapi: dict | None) -> str:
    openapi_section = ""
    if openapi:
        paths   = list((openapi.get("paths") or {}).keys())[:30]
        schemas = list(openapi.get("components", {}).get("schemas", {}).keys())[:20]
        openapi_section = "\n\n### OPENAPI ENDPOINTS\n" + "\n".join(paths)
        if schemas:
            openapi_section += f"\n\nSchemas: {', '.join(schemas)}"
    return (
        "Analyze this codebase and extract the security architecture.\n"
        "IMPORTANT: in the 'files' array, use exact relative paths as they appear "
        "in the ### FILE: headers — do not invent or abbreviate paths.\n\n"
        f"=== CODE ===\n{code_ctx}{openapi_section}\n\n"
        "Respond ONLY with the JSON object."
    )


def _call_gpt(messages: list[dict]) -> dict:
    """Single GPT-4o call — called via call_with_retry."""
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.1,
        response_format={"type": "json_object"},
    )
    log.info("Tokens — prompt: %d  completion: %d",
             resp.usage.prompt_tokens, resp.usage.completion_tokens)
    return parse_json_response(resp.choices[0].message.content, context="NB02")


user_msg  = _build_user_message(code_ctx, surface.get("openapi"))
messages  = [
    {"role": "system", "content": SYSTEM},
    {"role": "user",   "content": user_msg},
]

components = call_with_retry(
    _call_gpt,
    messages,
    max_retries=3,
    retry_prompt_suffix="Please respond with ONLY the JSON object, no markdown fences or explanation.",
    logger=log,
)
log.info("Extraction complete")

## 2.4 — Schema Validation

Verify the model returned all required top-level keys before proceeding.

In [ ]:
REQUIRED_KEYS = {"components", "data_flows", "trust_boundaries", "external_actors"}
missing_keys  = REQUIRED_KEYS - set(components.keys())
if missing_keys:
    raise ValueError(
        f"LLM response is missing required top-level keys: {missing_keys}.\n"
        "Re-run cell 2.3 or inspect the raw response."
    )

comp_ids = {c["id"] for c in components["components"]}

log.info("Components    : %d", len(components["components"]))
log.info("Data flows    : %d", len(components["data_flows"]))
log.info("Trust bounds  : %d", len(components["trust_boundaries"]))

print("\n── COMPONENTS ──")
for c in components["components"]:
    print(f"  [{c['type'].upper():12s}] {c['name']} — {c['description'][:70]}")

## 2.5 — Grounding Check

Components whose `files` array contains no real repo path are flagged and dropped. We use exact path matching against `valid_paths` (full relative paths). Ungrounded components are excluded from `components.json` so they never reach NB03 and waste API calls.

In [ ]:
def is_path_grounded(cited_file: str, valid_paths: set[str]) -> bool:
    """
    Return True if cited_file matches a real repo path.

    Accepts:
    - Exact full-path match (fastapi/routing.py)
    - Basename-only match for basenames >= 4 chars (routing.py)
    Rejects short/common tokens like 'db', 'api', 'io'.
    """
    if cited_file in valid_paths:
        return True
    # Basename fallback — only safe for names >= 4 chars
    if len(Path(cited_file).name) >= 4:
        basenames = {Path(p).name for p in valid_paths}
        return Path(cited_file).name in basenames
    return False


grounded_comps:   list[dict] = []
ungrounded_comps: list[dict] = []
grounding_issues: list[str]  = []

for comp in components["components"]:
    cited = comp.get("files", [])
    if not cited:
        grounding_issues.append(f"'{comp['name']}': no file citations")
        comp["grounded"] = False
        ungrounded_comps.append(comp)
        continue

    real_matches = [f for f in cited if is_path_grounded(f, valid_paths)]
    if not real_matches:
        grounding_issues.append(
            f"'{comp['name']}': cited files not in repo — {cited[:3]}"
        )
        comp["grounded"] = False
        ungrounded_comps.append(comp)
    else:
        comp["grounded"]      = True
        comp["matched_paths"] = real_matches
        grounded_comps.append(comp)

# Validate data flow references point to real (grounded) components
grounded_ids = {c["id"] for c in grounded_comps}
flow_issues: list[str] = []
grounded_flows: list[dict] = []

for flow in components["data_flows"]:
    src_ok = flow["from"] in grounded_ids
    dst_ok = flow["to"]   in grounded_ids
    if src_ok and dst_ok:
        grounded_flows.append(flow)
    else:
        if not src_ok:
            flow_issues.append(f"Flow source '{flow['from']}' is not a grounded component")
        if not dst_ok:
            flow_issues.append(f"Flow target '{flow['to']}' is not a grounded component")

if grounding_issues:
    print("\nComponent grounding issues (dropped):")
    for issue in grounding_issues:
        print(f"  ✗ {issue}")
if flow_issues:
    print("\nData flow issues (dropped):")
    for issue in flow_issues:
        print(f"  ✗ {issue}")

print(f"\nGrounded components : {len(grounded_comps)}")
print(f"Dropped components  : {len(ungrounded_comps)}")
print(f"Grounded flows      : {len(grounded_flows)}")
print(f"Dropped flows       : {len(components['data_flows']) - len(grounded_flows)}")

## 2.6 — Save `components.json`

Only grounded components and flows are saved. Ungrounded items are recorded in the metadata for audit purposes.

In [ ]:
output = {
    "components":      grounded_comps,
    "data_flows":      grounded_flows,
    "trust_boundaries": components.get("trust_boundaries", []),
    "external_actors":  components.get("external_actors", []),
    "metadata": {
        "run_id":               RUN_ID,
        "repo":                 surface["repo"],
        "total_components_raw": len(components["components"]),
        "grounded_components":  len(grounded_comps),
        "dropped_components":   len(ungrounded_comps),
        "grounding_issues":     grounding_issues,
        "flow_issues":          flow_issues,
        "model":                "gpt-4o",
    },
}

save_json(output, "components.json", run_id=RUN_ID)
print("Saved components.json (grounded only)")